# Test Loaders

Manual notebook checks for public functions in `src/loaders/data_loader.py`.

These checks are read-only. They call MongoDB `find` and `distinct` only.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

PROJECT_ROOT

WindowsPath('d:/vs code project/TechnicalResearch')

In [2]:
import pandas as pd
import polars as pl
from pymongo.collection import Collection

from config import load_mongo_settings
from loaders.data_loader import BAR_COLUMNS, get_collection, list_symbols, load_market_bars

EXPECTED_COLUMNS = ["_id", "time", "symbol", "open", "high", "low", "close", "volume"]
assert BAR_COLUMNS == EXPECTED_COLUMNS

## Configuration And Connection

In [3]:
settings = load_mongo_settings()

assert settings.uri
assert settings.database
assert settings.collection

collection = get_collection(settings)
assert isinstance(collection, Collection)
assert collection.database.name == settings.database
assert collection.name == settings.collection

collection

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'vn_market_data'), 'market_bars_raw')

## `list_symbols`

In [4]:
symbols = list_symbols(collection)

assert isinstance(symbols, list)
assert symbols == sorted(symbols)
assert all(isinstance(symbol, str) and symbol for symbol in symbols)
assert symbols, "No symbols found in MongoDB collection."

symbols[:10], len(symbols)

(['A32', 'AAA', 'AAH', 'AAM', 'AAS', 'AAT', 'AAV', 'ABB', 'ABC', 'ABI'], 1702)

## `load_market_bars` Pandas Output

In [5]:
sample_symbol = symbols[0]
bars_pd = load_market_bars(symbols=sample_symbol, limit=10, frame="pandas", collection=collection)

assert isinstance(bars_pd, pd.DataFrame)
assert list(bars_pd.columns) == EXPECTED_COLUMNS
assert len(bars_pd) <= 10
assert bars_pd["symbol"].eq(sample_symbol).all()
assert bars_pd["time"].is_monotonic_increasing

bars_pd

,_id,time,symbol,open,high,low,close,volume
0,6a26bf6ab3097ec8bfa07e6c,2018-10-24,A32,12.09,12.09,12.09,12.09,0
1,6a26bf6ab3097ec8bfa07e6d,2018-10-25,A32,12.09,12.09,12.09,12.09,0
2,6a26bf6ab3097ec8bfa07e6e,2018-10-26,A32,12.09,12.09,12.09,12.09,0
3,6a26bf6ab3097ec8bfa07e6f,2018-10-29,A32,12.09,12.09,12.09,12.09,0
4,6a26bf6ab3097ec8bfa07e70,2018-10-30,A32,12.09,12.09,12.09,12.09,0
5,6a26bf6ab3097ec8bfa07e71,2018-10-31,A32,12.09,12.09,12.09,12.09,0
6,6a26bf6ab3097ec8bfa07e72,2018-11-01,A32,12.09,12.09,12.09,12.09,0
7,6a26bf6ab3097ec8bfa07e73,2018-11-02,A32,12.09,12.09,12.09,12.09,0
8,6a26bf6ab3097ec8bfa07e74,2018-11-05,A32,12.09,12.09,12.09,12.09,0
9,6a26bf6ab3097ec8bfa07e75,2018-11-06,A32,12.09,12.09,12.09,12.09,0


## Symbol And Time Filters

In [6]:
if len(bars_pd) >= 2:
    start = bars_pd["time"].iloc[0]
    end = bars_pd["time"].iloc[-1]
    filtered = load_market_bars(
        symbols=[sample_symbol],
        start=start,
        end=end,
        limit=10,
        frame="pandas",
        collection=collection,
    )

    assert isinstance(filtered, pd.DataFrame)
    assert list(filtered.columns) == EXPECTED_COLUMNS
    assert filtered["symbol"].eq(sample_symbol).all()
    assert filtered["time"].ge(start).all()
    assert filtered["time"].lt(end).all()

    display(filtered)
else:
    print("Skipping time filter check because fewer than 2 rows were loaded.")

,_id,time,symbol,open,high,low,close,volume
0,6a26bf6ab3097ec8bfa07e6c,2018-10-24,A32,12.09,12.09,12.09,12.09,0
1,6a26bf6ab3097ec8bfa07e6d,2018-10-25,A32,12.09,12.09,12.09,12.09,0
2,6a26bf6ab3097ec8bfa07e6e,2018-10-26,A32,12.09,12.09,12.09,12.09,0
3,6a26bf6ab3097ec8bfa07e6f,2018-10-29,A32,12.09,12.09,12.09,12.09,0
4,6a26bf6ab3097ec8bfa07e70,2018-10-30,A32,12.09,12.09,12.09,12.09,0
5,6a26bf6ab3097ec8bfa07e71,2018-10-31,A32,12.09,12.09,12.09,12.09,0
6,6a26bf6ab3097ec8bfa07e72,2018-11-01,A32,12.09,12.09,12.09,12.09,0
7,6a26bf6ab3097ec8bfa07e73,2018-11-02,A32,12.09,12.09,12.09,12.09,0
8,6a26bf6ab3097ec8bfa07e74,2018-11-05,A32,12.09,12.09,12.09,12.09,0


## `load_market_bars` Polars Output

In [7]:
bars_pl = load_market_bars(symbols=sample_symbol, limit=10, frame="polars", collection=collection)

assert isinstance(bars_pl, pl.DataFrame)
assert bars_pl.columns == EXPECTED_COLUMNS
assert bars_pl.height <= 10
assert bars_pl.select(pl.col("symbol").eq(sample_symbol).all()).item()

bars_pl

_id,time,symbol,open,high,low,close,volume
str,str,str,f64,f64,f64,f64,i64
"""6a26bf6ab3097ec8bfa07e6c""","""2018-10-24""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e6d""","""2018-10-25""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e6e""","""2018-10-26""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e6f""","""2018-10-29""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e70""","""2018-10-30""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e71""","""2018-10-31""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e72""","""2018-11-01""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e73""","""2018-11-02""","""A32""",12.09,12.09,12.09,12.09,0
"""6a26bf6ab3097ec8bfa07e74""","""2018-11-05""","""A32""",12.09,12.09,12.09,12.09,0


## Invalid Frame Validation

In [8]:
try:
    load_market_bars(symbols=sample_symbol, limit=1, frame="invalid", collection=collection)  # type: ignore[arg-type]
except ValueError as exc:
    assert "frame must be either" in str(exc)
else:
    raise AssertionError("load_market_bars should reject unsupported frame values.")

print("Invalid frame check passed.")

Invalid frame check passed.


## Summary

In [9]:
print("All loader checks passed.")

All loader checks passed.
